# Mode 1 Domain Initializers Demo

This notebook exercises the current simpler Mode 1 initialization surface in 1D, 2D, and 3D.

It shows how to:
- initialize practical domains through `initialize_mode1_domain`
- optimize fixed-topology coordinates with `run_mode1_workflow`
- export `.msh`, metrics, history, snapshots, and visualization payloads
- optionally view results with Gmsh or Matplotlib


In [ ]:
from pathlib import Path
import json

import jax.numpy as jnp

from topojax import (
    build_mode1_visualization_payload,
    initialize_mode1_domain,
    load_gmsh_msh,
    run_mode1_workflow,
    visualize_mode1_result,
)

OUTPUT_ROOT = Path("outputs/topo/notebooks/mode1_domain_initializers_demo")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Set to True to open interactive viewers.
LAUNCH_GMSH = False
SHOW_MATPLOTLIB = False

print(f"writing outputs under: {OUTPUT_ROOT.resolve()}")

In [ ]:
def summarize_artifacts(name, run):
    imported = load_gmsh_msh(run.artifacts["mesh"])
    summary = {
        "name": name,
        "points_shape": tuple(run.result.points.shape),
        "elements_shape": tuple(run.result.topology.elements.shape),
        "primary_element_kind": imported.primary_element_kind,
        "mesh": str(run.artifacts["mesh"]),
        "viewer_payload": str(run.artifacts["viewer_payload"]),
        "metrics_json": str(run.artifacts["metrics_json"]),
        "history_json": str(run.artifacts["history_json"]),
        "topo_snapshot": str(run.artifacts["topo_snapshot"]),
    }
    if "stl" in run.artifacts:
        summary["stl"] = str(run.artifacts["stl"])
    return summary


def run_case(name, kind, *, workflow_kwargs=None, view_backend=None, **domain_kwargs):
    workflow_kwargs = dict(workflow_kwargs or {})
    domain = initialize_mode1_domain(kind, **domain_kwargs)
    run = run_mode1_workflow(
        domain,
        output_dir=OUTPUT_ROOT / name,
        prefix=name,
        steps=workflow_kwargs.pop("steps", 4),
        step_size=workflow_kwargs.pop("step_size", 0.01),
        diagnostics_every=workflow_kwargs.pop("diagnostics_every", 2),
        launch_gmsh=workflow_kwargs.pop("launch_gmsh", False),
        **workflow_kwargs,
    )

    payload = build_mode1_visualization_payload(run.result.points, run.result.topology, title=name)
    summary = summarize_artifacts(name, run)
    summary["topology_kind"] = payload["topology_kind"]

    if LAUNCH_GMSH and view_backend in (None, "gmsh"):
        visualize_mode1_result(run.result, backend="gmsh", mesh_path=run.artifacts["mesh"], wait=False)
    elif SHOW_MATPLOTLIB and view_backend == "matplotlib":
        visualize_mode1_result(run.result, backend="matplotlib")

    return run, summary


## 1D

Intervals and polylines both map to fixed line-element connectivity for Mode 1.

In [ ]:
runs = {}
summaries = []

runs["interval"], summary = run_case("interval", "interval", n=16)
summaries.append(summary)

polyline_points = jnp.asarray([
    [0.0, 0.0],
    [0.2, 0.02],
    [0.5, -0.03],
    [0.8, 0.01],
    [1.0, 0.0],
])
runs["polyline"], summary = run_case("polyline", "polyline", points=polyline_points)
summaries.append(summary)

summaries[-2:]

## 2D

The current simple surface path covers squares, mapped squares, polygons, and polygon-derived quads.

In [ ]:
runs["square_quad"], summary = run_case("square_quad", "square", nx=18, ny=12, family="quad")
summaries.append(summary)

def mapped_square(xi_eta):
    x = xi_eta[:, 0]
    y = xi_eta[:, 1]
    return jnp.stack([x + 0.18 * y, y + 0.08 * jnp.sin(jnp.pi * x)], axis=1)

runs["mapped_square_quad"], summary = run_case(
    "mapped_square_quad",
    "square",
    nx=18,
    ny=12,
    family="quad",
    map_fn=mapped_square,
)
summaries.append(summary)

outer = jnp.asarray([[0.0, 0.0], [2.0, 0.0], [2.0, 1.0], [1.2, 1.0], [1.2, 2.0], [0.0, 2.0]])
hole = jnp.asarray([[0.45, 0.45], [0.95, 0.45], [0.95, 0.95], [0.45, 0.95]])

runs["polygon_tri"], summary = run_case(
    "polygon_tri",
    "polygon",
    outer_boundary=outer,
    holes=[hole],
    target_edge_size=0.18,
)
summaries.append(summary)

runs["polygon_quad"], summary = run_case(
    "polygon_quad",
    "polygon-quad",
    outer_boundary=outer,
    holes=[hole],
    target_edge_size=0.20,
)
summaries.append(summary)

summaries[-4:]

## 3D

The current simple volume/surface path covers boxes, sphere surfaces, and sphere volumes.

In [ ]:
runs["box"], summary = run_case(
    "box",
    "box",
    nx=7,
    ny=5,
    nz=6,
    workflow_kwargs={"export_stl_surface": True, "step_size": 0.012},
)
summaries.append(summary)

runs["sphere_surface"], summary = run_case(
    "sphere_surface",
    "sphere-surface",
    center=jnp.asarray([0.5, 0.5, 0.5]),
    radius=0.42,
    n_lat=7,
    n_lon=14,
    workflow_kwargs={"step_size": 0.01},
)
summaries.append(summary)

runs["sphere_volume"], summary = run_case(
    "sphere_volume",
    "sphere-volume",
    center=jnp.asarray([0.5, 0.5, 0.5]),
    radius=0.42,
    nx=9,
    ny=9,
    nz=9,
    workflow_kwargs={"step_size": 0.012},
)
summaries.append(summary)

summaries[-3:]

## Write A Summary File

This writes one machine-readable summary file listing the outputs created by the notebook.

In [ ]:
summary_path = OUTPUT_ROOT / "artifact_summary.json"
summary_path.write_text(json.dumps(summaries, indent=2), encoding="utf-8")
summary_path

In [ ]:
json.loads(summary_path.read_text(encoding="utf-8"))